# Model Evaluation — REPSOL Industrial Sound Classification

Evaluate and compare trained models on the REPSOL test set.

**Models compared:**
- `EfficientNet-B0` (fine-tuned, `efficientnet_best_01.pth` and `efficientnet_best_02.pth`)
- `PreTrained_model` (`PreTrained_model_best_01.pth`)

**Metrics computed:**
- Accuracy, Precision, Recall, F1 (weighted & macro)
- Per-class Precision / Recall / F1 / Support
- Confusion matrix
- ROC-AUC (macro, one-vs-rest)
- Top-2 accuracy
- Confidence distribution (correct vs wrong)
- Learning curves
- Misclassification analysis

## 0. Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path
REPO_ROOT = Path(r"D:\Work\Internships\INMAR\REPSOL")
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "src"))

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    ConfusionMatrixDisplay,
)
from sklearn.preprocessing import label_binarize

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"Repo root: {REPO_ROOT}")

## 1. Paths

In [ ]:
SPECT_DIR      = REPO_ROOT / "Data" / "Spectrograms"
ANNOTATIONS    = REPO_ROOT / "Data" / "Annotations"
MODELS_OUTPUT  = REPO_ROOT / "Models_output"
EVAL_OUT       = REPO_ROOT / "outputs" / "evaluation"
EVAL_OUT.mkdir(parents=True, exist_ok=True)

CHECKPOINTS = {
    "EfficientNet_01": MODELS_OUTPUT / "efficientnet_best_01.pth",
    "EfficientNet_02": MODELS_OUTPUT / "efficientnet_best_02.pth",
    "PreTrained_01":   MODELS_OUTPUT / "PreTrained_model_best_01.pth",
}

HISTORIES = {
    "EfficientNet_01": MODELS_OUTPUT / "efficientnet_best_training_history_01.csv",
    "EfficientNet_02": MODELS_OUTPUT / "efficientnet_best_02_training_history.csv",
}

for name, path in CHECKPOINTS.items():
    status = "✓" if path.exists() else "✗ MISSING"
    print(f"  {status}  {name}: {path.name}")

## 2. Load Test DataLoader

In [ ]:
from src.dataloaders import get_dataloaders

_, _, test_loader = get_dataloaders(
    SPECT_DIR,
    batch_size=32,
    num_workers=0,
    pin_memory=False,
    persistent_workers=False,
)

CLASS_NAMES = test_loader.dataset.classes
NUM_CLASSES = len(CLASS_NAMES)
print(f"Test samples : {len(test_loader.dataset)}")
print(f"Classes ({NUM_CLASSES}):")
for i, c in enumerate(CLASS_NAMES):
    print(f"  {i}: {c}")

## 3. Inference Helper

In [ ]:
def run_inference(model, loader, device):
    """Return y_true, y_pred, y_probs (softmax) for the full loader."""
    model.eval()
    all_true, all_pred, all_probs = [], [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            logits = model(x)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = probs.argmax(axis=1)
            all_true.extend(y.numpy())
            all_pred.extend(preds)
            all_probs.append(probs)
    return (
        np.array(all_true),
        np.array(all_pred),
        np.vstack(all_probs),
    )

## 4. Metrics Helper

In [ ]:
def compute_metrics(y_true, y_pred, y_probs, class_names):
    n_classes = len(class_names)

    # --- scalar metrics ---
    acc = accuracy_score(y_true, y_pred)

    prec_w, rec_w, f1_w, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    prec_m, rec_m, f1_m, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )

    # top-2 accuracy
    top2_correct = sum(
        y_true[i] in np.argsort(y_probs[i])[-2:]
        for i in range(len(y_true))
    )
    top2_acc = top2_correct / len(y_true)

    # ROC-AUC (one-vs-rest, macro)
    try:
        y_bin = label_binarize(y_true, classes=range(n_classes))
        roc_auc = roc_auc_score(y_bin, y_probs, average="macro", multi_class="ovr")
    except Exception:
        roc_auc = float("nan")

    # mean confidence on correct vs wrong predictions
    max_conf = y_probs.max(axis=1)
    correct_mask = y_true == y_pred
    mean_conf_correct = max_conf[correct_mask].mean() if correct_mask.any() else float("nan")
    mean_conf_wrong   = max_conf[~correct_mask].mean() if (~correct_mask).any() else float("nan")

    # per-class metrics
    prec_pc, rec_pc, f1_pc, sup_pc = precision_recall_fscore_support(
        y_true, y_pred, average=None, zero_division=0, labels=range(n_classes)
    )
    per_class_df = pd.DataFrame({
        "class":     class_names,
        "precision": prec_pc,
        "recall":    rec_pc,
        "f1":        f1_pc,
        "support":   sup_pc.astype(int),
    })

    cm = confusion_matrix(y_true, y_pred, labels=range(n_classes))

    return {
        "accuracy":           acc,
        "precision_weighted": prec_w,
        "recall_weighted":    rec_w,
        "f1_weighted":        f1_w,
        "precision_macro":    prec_m,
        "recall_macro":       rec_m,
        "f1_macro":           f1_m,
        "top2_accuracy":      top2_acc,
        "roc_auc_macro":      roc_auc,
        "mean_conf_correct":  mean_conf_correct,
        "mean_conf_wrong":    mean_conf_wrong,
        "per_class":          per_class_df,
        "confusion_matrix":   cm,
        "y_true":             y_true,
        "y_pred":             y_pred,
        "y_probs":            y_probs,
    }

## 5. Evaluate All Models

In [ ]:
from src.PreTrained_model.load_pretrained import load_model

results = {}

for model_name, ckpt_path in CHECKPOINTS.items():
    if not ckpt_path.exists():
        print(f"  ✗ Skipping {model_name} — checkpoint not found")
        continue

    print(f"\nEvaluating {model_name} ...")
    model, _ = load_model(
        checkpoint_path=str(ckpt_path),
        model_name="efficientnet",
        num_classes=NUM_CLASSES,
        device=DEVICE,
    )

    y_true, y_pred, y_probs = run_inference(model, test_loader, DEVICE)
    metrics = compute_metrics(y_true, y_pred, y_probs, CLASS_NAMES)
    results[model_name] = metrics

    print(f"  Accuracy        : {metrics['accuracy']:.4f}")
    print(f"  F1 (weighted)   : {metrics['f1_weighted']:.4f}")
    print(f"  F1 (macro)      : {metrics['f1_macro']:.4f}")
    print(f"  Top-2 Accuracy  : {metrics['top2_accuracy']:.4f}")
    print(f"  ROC-AUC (macro) : {metrics['roc_auc_macro']:.4f}")

print("\nDone.")

## 6. Summary Table — All Models

In [ ]:
SCALAR_KEYS = [
    "accuracy", "precision_weighted", "recall_weighted", "f1_weighted",
    "precision_macro", "recall_macro", "f1_macro",
    "top2_accuracy", "roc_auc_macro",
    "mean_conf_correct", "mean_conf_wrong",
]

summary = pd.DataFrame(
    {name: {k: m[k] for k in SCALAR_KEYS} for name, m in results.items()}
).T.round(4)

display(summary)

## 7. Confusion Matrices

In [ ]:
SHORT_NAMES = [c.split()[0] for c in CLASS_NAMES]  # shorten long class names for display

for model_name, m in results.items():
    cm = m["confusion_matrix"]
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    # Raw counts
    disp = ConfusionMatrixDisplay(cm, display_labels=SHORT_NAMES)
    disp.plot(ax=axes[0], cmap="Blues", colorbar=False, xticks_rotation=45)
    axes[0].set_title(f"{model_name} — Counts")

    # Normalised (recall per class)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)
    disp2 = ConfusionMatrixDisplay(cm_norm, display_labels=SHORT_NAMES)
    disp2.plot(ax=axes[1], cmap="Blues", colorbar=True, xticks_rotation=45)
    axes[1].set_title(f"{model_name} — Normalised (recall)")

    fig.suptitle(f"Confusion Matrix — {model_name}", fontsize=14)
    fig.tight_layout()
    fig.savefig(EVAL_OUT / f"confusion_matrix_{model_name}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 8. Per-Class Metrics

In [ ]:
for model_name, m in results.items():
    df = m["per_class"].copy()
    df["class"] = df["class"].str[:35]  # truncate long names

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, col, color in zip(axes, ["precision", "recall", "f1"], ["C0", "C1", "C2"]):
        df.sort_values(col).plot.barh(x="class", y=col, ax=ax, color=color, legend=False)
        ax.set_xlim(0, 1)
        ax.set_xlabel(col)
        ax.set_title(col.capitalize())
        ax.axvline(0.5, color="red", linestyle="--", linewidth=0.8, alpha=0.6)

    fig.suptitle(f"Per-Class Metrics — {model_name}", fontsize=13)
    fig.tight_layout()
    fig.savefig(EVAL_OUT / f"per_class_{model_name}.png", dpi=150, bbox_inches="tight")
    plt.show()
    display(df.set_index("class").round(3))

## 9. Model Comparison — F1 per Class

In [ ]:
f1_compare = pd.DataFrame(
    {name: m["per_class"].set_index("class")["f1"] for name, m in results.items()}
)
f1_compare.index = f1_compare.index.str[:35]

ax = f1_compare.plot.barh(figsize=(12, 7))
ax.set_xlim(0, 1)
ax.axvline(0.5, color="red", linestyle="--", linewidth=0.8, alpha=0.6)
ax.set_title("F1 Score per Class — Model Comparison")
ax.set_xlabel("F1")
plt.tight_layout()
plt.savefig(EVAL_OUT / "f1_comparison_per_class.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Confidence Distribution

In [ ]:
for model_name, m in results.items():
    y_true  = m["y_true"]
    y_pred  = m["y_pred"]
    y_probs = m["y_probs"]
    max_conf = y_probs.max(axis=1)
    correct  = y_true == y_pred

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Histogram: correct vs wrong
    axes[0].hist(max_conf[correct],  bins=20, alpha=0.7, label="Correct",  color="C2")
    axes[0].hist(max_conf[~correct], bins=20, alpha=0.7, label="Wrong",    color="C3")
    axes[0].set_xlabel("Max softmax confidence")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Confidence distribution")
    axes[0].legend()

    # Strip plot: confidence by class
    df_conf = pd.DataFrame({
        "class":      [CLASS_NAMES[i][:30] for i in y_true],
        "confidence": max_conf,
        "correct":    correct,
    })
    class_order = df_conf.groupby("class")["confidence"].median().sort_values().index.tolist()
    sns.stripplot(
        data=df_conf, y="class", x="confidence", hue="correct",
        order=class_order, dodge=False, alpha=0.5, jitter=0.25,
        palette={True: "C2", False: "C3"}, ax=axes[1]
    )
    axes[1].set_title("Confidence by class")
    axes[1].set_xlabel("Max softmax confidence")
    axes[1].legend(title="Correct")

    fig.suptitle(f"Confidence — {model_name}", fontsize=13)
    fig.tight_layout()
    fig.savefig(EVAL_OUT / f"confidence_{model_name}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 11. Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for model_name, hist_path in HISTORIES.items():
    if not hist_path.exists():
        print(f"  History not found: {hist_path.name}")
        continue
    df = pd.read_csv(hist_path)
    acc_col = "train_acc" if "train_acc" in df.columns else "train_accuracy"
    val_col = "val_acc"   if "val_acc"   in df.columns else "val_accuracy"

    axes[0].plot(df["epoch"], df[acc_col],  label=f"{model_name} train")
    axes[0].plot(df["epoch"], df[val_col],  label=f"{model_name} val", linestyle="--")
    axes[1].plot(df["epoch"], df["train_loss"], label=f"{model_name} train")
    axes[1].plot(df["epoch"], df["val_loss"],   label=f"{model_name} val",   linestyle="--")

axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].set_title("Loss");     axes[1].set_xlabel("Epoch"); axes[1].legend()
fig.suptitle("Learning Curves", fontsize=13)
fig.tight_layout()
fig.savefig(EVAL_OUT / "learning_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Misclassification Analysis

In [ ]:
for model_name, m in results.items():
    y_true  = m["y_true"]
    y_pred  = m["y_pred"]
    y_probs = m["y_probs"]

    mask = y_true != y_pred
    df_mis = pd.DataFrame({
        "true":       [CLASS_NAMES[i][:40] for i in y_true[mask]],
        "predicted":  [CLASS_NAMES[i][:40] for i in y_pred[mask]],
        "confidence": y_probs[mask].max(axis=1).round(3),
    })

    # Most common confusions
    top_confusions = (
        df_mis.groupby(["true", "predicted"])
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .head(10)
    )

    print(f"\n{'='*60}")
    print(f"{model_name} — {mask.sum()} misclassified out of {len(y_true)}")
    print(f"{'='*60}")
    display(top_confusions.reset_index(drop=True))

    df_mis.to_csv(EVAL_OUT / f"misclassified_{model_name}.csv", index=False)

## 13. Classification Report (Full)

In [ ]:
for model_name, m in results.items():
    print(f"\n{'='*60}")
    print(f"Classification Report — {model_name}")
    print(f"{'='*60}")
    report = classification_report(
        m["y_true"], m["y_pred"],
        target_names=CLASS_NAMES,
        zero_division=0
    )
    print(report)

## 14. Save Summary CSV

In [ ]:
summary.to_csv(EVAL_OUT / "summary_metrics.csv")
print(f"Results saved to {EVAL_OUT}")
for f in sorted(EVAL_OUT.iterdir()):
    print(f"  {f.name}")